**BASE**

In [ ]:
# importamos paquetes
import pandas as pd
import numpy as np
np.random.seed(42)

# número de clientes
num_clientes = 2000

**TABLA HOGAR**

In [ ]:
# hogar_id
hogar_id = ["hogar_" + str(i).zfill(4) for i in range(1, num_clientes + 1)]

# tipo_inmueble
tipo_inmueble = np.random.choice(
    a=['casa', 'departamento'],
    size=num_clientes,
    replace=True, p=[0.65, 0.35])

# zona_fria
zona_fria = np.random.choice(
    a=['si', 'no'],
    size=num_clientes,
    replace=True, p=[0.4, 0.6])

# calidad_aislamiento
calidad_aislamiento = ['muy baja', 'baja', 'media', 'alta', 'muy alta']
calidad_aislamiento = np.random.choice(
    a=calidad_aislamiento,
    size=num_clientes,
    replace=True,
    p=[0.15, 0.25, 0.4, 0.1, 0.1])

# fuente_calefaccion
fuente_calefaccion = ['electricidad', 'solar', 'otros']
fuente_calefaccion = np.random.choice(
    a=fuente_calefaccion,
    size=num_clientes,
    replace=True,
    p=[0.5, 0.3, 0.2])

# fuente_agua
fuente_agua = ['electricidad', 'solar', 'otros']
fuente_agua = np.random.choice(
    a=fuente_agua,
    size=num_clientes,
    replace=True,
    p=[0.5, 0.3, 0.2])

# energia_solar (por cliente)
energia_solar = np.where((fuente_calefaccion == 'solar') | (fuente_agua == 'solar'), 'si', 'no')


In [ ]:
# dataframe

df_hogar = pd.DataFrame({
    'hogar_id': hogar_id,
    'tipo_inmueble': tipo_inmueble,
    'zona_fria': zona_fria,
    'calidad_aislamiento': calidad_aislamiento,
    'fuente_calefaccion': fuente_calefaccion,
    'fuente_agua': fuente_agua,
    'energia_solar': energia_solar
})

In [ ]:
df_hogar.head(3)

**TABLA CONSUMO**

In [ ]:
energia_solar_consumo = df_hogar["energia_solar"].values
zona_fria_consumo = df_hogar["zona_fria"].values

# consumo_id
consumo_id = ["consumo_" + str(i).zfill(4) for i in range(1, num_clientes + 1)]

# consumo_total
consumo_total = np.random.randint(
    low=30,
    high=1500,
    size=num_clientes
)

# generacion_solar_kwh
generacion_solar_kwh = np.where(
    energia_solar_consumo == "si",
    np.random.randint(50, 800, size=num_clientes),
    0
)

# consumo_solar_kwh
consumo_solar_kwh = np.array([
    np.random.uniform(0, min(generacion, consumo))
    for generacion, consumo in zip(generacion_solar_kwh, consumo_total)
])

# consumo_electrico_kwh
consumo_electrico_kwh = consumo_total - consumo_solar_kwh

# tarifa_kwh
tarifa_kwh = 0.75

# costo_bruto
costo_bruto = consumo_electrico_kwh * tarifa_kwh

# descuento_zona_fría
descuento_zona_fria = np.where(
    zona_fria_consumo == "si",
    costo_bruto * 0.30,
    0
)

# precio_final
precio_total = costo_bruto - descuento_zona_fria

In [ ]:
df_consumo = pd.DataFrame({
    'hogar_id': df_hogar['hogar_id'],
    'consumo_id': consumo_id,
    'consumo_total_kwh': consumo_total.round(2),
    'consumo_electrico_kwh': consumo_electrico_kwh.round(2),
    'generacion_solar_kwh': generacion_solar_kwh,
    'consumo_solar_kwh': consumo_solar_kwh.round(2),
    'precio_total': precio_total.round(2)
})

In [ ]:
df_consumo.head(3)

**TABLA EQUIPAMIENTO**

In [ ]:
# equipamiento_id
equipamiento_id = ["equipamiento_" + str(i).zfill(4) for i in range(1, num_clientes + 1)]

# cantidad_equipos
cantidad_equipos = np.random.randint(
    low=1,
    high=11,
    size=num_clientes)

In [ ]:
# clase A - bajo consumo
# clase B - consumo medio
# clase C - alto consumo

# generar proporciones aleatorias para las 3 clases
proporciones_clases = np.random.dirichlet(
    alpha=[1, 2, 3],
    size=num_clientes
)

# convertir a cantidades enteras y repartir los restantes
cantidades = np.floor(proporciones_clases * cantidad_equipos[:, None]).astype(int)
restantes = cantidad_equipos - cantidades.sum(axis=1)

for i in range(num_clientes):
  if restantes[i] > 0:
    decimales = proporciones_clases[i] * cantidad_equipos[i] - cantidades[i]
    idx = np.argsort(-decimales)[:restantes[i]]
    cantidades[i, idx] += 1

# separar las proporciones de las clases
clase_a = cantidades[:, 0]
clase_b = cantidades[:, 1]
clase_c = cantidades[:, 2]

In [ ]:
# submetering_1: heladera, lavavajilla, horno, microondas
# submetering_2: lavarropas, secarropas
# submetering_3: aires acondicionados y calefacción
# submetering_4: toda la iluminación, interior y exterior
# submetering_5: TV, computadoras, routers
# submetering_6: cargas que no encajen en las categorías anteriores, como la carga del vehículo eléctrico

# generar proporciones aleatorias para los 6 submetering
proporciones = np.random.dirichlet(
    alpha=[2, 1.5, 3, 1, 1.5, 1],
    size=num_clientes
)

# separar las proporciones
p1 = proporciones[:, 0]
p2 = proporciones[:, 1]
p3 = proporciones[:, 2]
p4 = proporciones[:, 3]
p5 = proporciones[:, 4]
p6 = proporciones[:, 5]

# consumo total del hogar
consumo_total = df_consumo["consumo_total_kwh"].values

# asignar consumo a cada categoría
submetering_1 = consumo_total * p1
submetering_2 = consumo_total * p2
submetering_3 = consumo_total * p3
submetering_4 = consumo_total * p4
submetering_5 = consumo_total * p5
submetering_6 = consumo_total * p6

In [ ]:
# dataframe

df_equipamientos = pd.DataFrame({
    "equipamiento_id": equipamiento_id,
    "hogar_id": df_hogar["hogar_id"],
    "cantidad_equipos": cantidad_equipos,
    "clase_a": clase_a,
    "clase_b": clase_b,
    "clase_c": clase_c,
    "submetering_1": submetering_1.round(2),
    "submetering_2": submetering_2.round(2),
    "submetering_3": submetering_3.round(2),
    "submetering_4": submetering_4.round(2),
    "submetering_5": submetering_5.round(2),
    "submetering_6": submetering_6.round(2),
    "consumo_total": df_consumo['consumo_total_kwh']
})

In [ ]:
df_equipamientos.head()

**Clasificación de perfil**

In [ ]:
# tres tablas unidas
df_clasificacion = pd.merge(df_hogar, df_consumo, on='hogar_id', how='outer')
df_clasificacion = pd.merge(df_clasificacion, df_equipamientos.drop(columns=['consumo_total']), on='hogar_id', how='outer')

df_clasificacion = df_clasificacion.drop(columns=['equipamiento_id', 'consumo_id'])

df_clasificacion.head(3)

In [ ]:
# dimensión consumo eléctrico

def score_consumo(df):

  # max, min y 50% obtenidas de df_clasificacion['consumo_total_kwh].describe()
  condiciones = [
        df["consumo_total_kwh"] <= 250,
        (df["consumo_total_kwh"] > 30) & (df["consumo_total_kwh"] <= 801),
        df["consumo_total_kwh"] > 801
    ]

  puntajes = [100, 67, 33]

  return np.select(condiciones, puntajes, default=0)

In [ ]:
def score_equipamiento(df):

  # max, min y 50% obtenidas de df_clasificacion['cantidad_equipos'].describe()
  condiciones = [
        df["cantidad_equipos"] <= 1,
        (df["cantidad_equipos"] > 1) & (df["cantidad_equipos"] <= 6),
        df["cantidad_equipos"] > 6
    ]

  puntajes = [100, 67, 33]

  return np.select(condiciones, puntajes, default=0)

In [ ]:
# dimensión eficiencia del hogar
# acá se reparte los 100 puntos por dimensión en las 3 variables distintas: 40, 30 y 30, respectivamente

def score_eficiencia(df):

  aislamiento = df['calidad_aislamiento'].map({
      'muy alta': 40,
      'alta': 35,
      'media': 24,
      'baja': 16,
      'muy baja': 8
  }).fillna(0)

  calefaccion = df['fuente_calefaccion'].map({
      'solar': 30,
      'electricidad': 5,
      'otros': 15
  }).fillna(0)

  agua = df['fuente_agua'].map({
      'solar': 30,
      'electricidad': 5,
      'otros': 15
  }).fillna(0)

  return aislamiento + calefaccion + agua

In [ ]:
# dimensión energia renovable

def score_renovable(df):
  score = np.where(df['energia_solar'] == 'si', 100, 50)

  return score

In [ ]:
# dimensión contexto del hogar

def score_contexto(df):
  inmueble = df['tipo_inmueble'].map({
      'casa': 40,
      'departamento': 60
  }).fillna(0)

  return inmueble

In [ ]:
# suma del puntaje obtenido

def calcular_puntaje(df):

    return (
        score_consumo(df) * 0.35 +
        score_equipamiento(df) * 0.25 +
        score_eficiencia(df) * 0.25 +
        score_renovable(df) * 0.10 +
        score_contexto(df) * 0.05
    )

In [ ]:
# asignacion de categoria

def asignar_categoria(df):

  df = df.copy()
  df['puntaje'] = calcular_puntaje(df)

  condiciones = [
      df['puntaje'] >= 72,
      (df['puntaje'] < 80) & (df['puntaje'] >= 49),
      (df['puntaje'] < 49)
  ]

  categorias = [
      'eficiente',
      'moderado',
      'ineficiente'
  ]

  df['categoria'] = np.select(condiciones, categorias, default='')

  return df

In [ ]:
df_final = asignar_categoria(df_clasificacion)
df_final = df_final[df_final['categoria'] != '']
df_final.to_json('database_beta.json', orient='records', indent=4)
print(df_final.groupby('categoria')['consumo_total_kwh'].mean())

In [ ]:
df_final.head(2)

# **BASE FINAL** (fuera de servicio momentaneamente)

In [ ]:
#df_final = pd.merge(df_hogar, df_consumo, on='hogar_id', how='outer')
#df_final = pd.merge(df_final, df_equipamientos.drop(columns=['consumo_total']), on='hogar_id', how='outer')

#df_final = df_final.drop(columns=['equipamiento_id', 'consumo_id'])

#df_final.head(3)

In [ ]:
#df_final.to_json('database_beta.json', orient='columns', indent=4)
#df_final.to_csv('database_beta.csv', index=False, sep=';')

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

features_categoricas = [
    'tipo_inmueble', 'zona_fria', 'calidad_aislamiento',
    'fuente_calefaccion', 'fuente_agua', 'energia_solar'
]

features_numericas = [
    'consumo_electrico_kwh', 'cantidad_equipos', 'submetering_1',
    'submetering_2', 'submetering_3', 'submetering_4',
    'submetering_5', 'submetering_6'
]

X = df_final[features_categoricas + features_numericas]
y = df_final['categoria']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocesador = ColumnTransformer(
    transformers=[
        ('numericas', StandardScaler(), features_numericas),
        ('categoricas', OneHotEncoder(handle_unknown='ignore'), features_categoricas)
    ]
)

modelo_eficiencia = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('clasificador', RandomForestClassifier(
        n_estimators=100, n_jobs=-1, class_weight='balanced', random_state=42
    ))
])

modelo_eficiencia.fit(X_train, y_train)
y_pred = modelo_eficiencia.predict(X_test)
print(classification_report(y_test, y_pred))
joblib.dump(modelo_eficiencia, 'modelo_eficiencia_v1.joblib')